This notebook is for reading and processing the data needed for the other notebooks in this folder.

In [ ]:
import os

# On Colab, mount Google Drive and switch to the notebook folder.
# Locally there is nothing to mount: Jupyter already runs in this folder.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # don't attempt to remount if the drive is already mounted
    if not os.path.exists("/content/mnt/MyDrive"):
        drive.mount("mnt")
    %cd '/content/mnt/MyDrive/Colab Notebooks/ERAD-nowcasting-course-2026/notebooks/exercise_notebooks/'

# run the previous notebook to configure the environment
%run helper_setup_pip.ipynb

In [ ]:
import xarray as xr
import numpy as np

OSN_ENDPOINT = "https://umn1.osn.mghpcc.org"
BUCKET = "nexrad-arco"
composite_url = f"s3://{BUCKET}/composite.zarr"

ds = xr.open_zarr(
    composite_url,
    storage_options={"anon": True, "client_kwargs": {"endpoint_url": OSN_ENDPOINT}},
    consolidated=False,
    decode_coords="all",
)

precip = ds.rain_rate.values
if ds.spatial_ref.attrs["grid_mapping_name"] != "lambert_azimuthal_equal_area":
  raise ValueError("Expected the grid mapping of the RHMSS data to be lambert_azimuthal_equal_area")
timestamps = np.array([timestamp.item() for timestamp in ds.vcp_time.values.astype('datetime64[us]')])
xpixelsize = float(ds.x.values[1] - ds.x.values[0])
ypixelsize = float(ds.y.values[1] - ds.y.values[0])
x1 = float(ds.x.values[0] - xpixelsize/2)
x2 = float(ds.x.values[-1] + xpixelsize/2)
y1 = float(ds.y.values[0] - ypixelsize/2)
y2 = float(ds.y.values[-1] + ypixelsize/2)
yorigin = "lower" if ypixelsize > 0 else "upper"
timestep = int((timestamps[1] - timestamps[0]).total_seconds() / 60)
metadata = {
  'accutime': timestep,
  'cartesian_unit': 'm',
  'institution': 'RHMSS',
  'projection': f'+proj=laea +lat_0={float(ds.spatial_ref.attrs["latitude_of_projection_origin"])} +lon_0={float(ds.spatial_ref.attrs["longitude_of_projection_origin"])} +x_0={float(ds.spatial_ref.attrs["false_easting"])} +y_0={float(ds.spatial_ref.attrs["false_northing"])}',
  'threshold': 0.01,
  'timestamps': timestamps,
  'transform': None,
  'unit': 'mm/h',
  'x1': x1,
  'x2': x2,
  'xpixelsize': xpixelsize,
  'y1': y1 if yorigin == "lower" else y2,
  'y2': y2 if yorigin == "lower" else y1,
  'yorigin': yorigin,
  'ypixelsize': abs(ypixelsize),
  'zerovalue': 0.0,
  'zr_a': 200.0, # Marshall-Palmer
  'zr_b': 1.6 # Marshall-Palmer
}